# NRW Borehole Stratigraphy Retrieval and 3D Visualization

**Author:** Vinicius Inojosa  
**Contact:** vinicius.inojosa@thga.de

This notebook demonstrates how to retrieve borehole data from the public NRW borehole database, extract stratigraphic information, convert lithological layers into regular depth intervals, and visualize them in 3D.

Data source: https://www.bohrungen.nrw.de/

The area of interest can be provided either as:

1. a WKT polygon, or  
2. a local shapefile path.

The WFS request is based on the bounding box of the provided area.


## 1. Imports and configuration


In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np
import requests
import xml.etree.ElementTree as ET
import plotly.express as px

from shapely import wkt as shapely_wkt
from shapely.geometry import Point

# Output files
OUTPUT_GML = Path("borehole_headers.gml")
OUTPUT_GPKG = Path("borehole_layers_3d.gpkg")
OUTPUT_HTML = Path("boreholes_3d.html")

# WFS endpoints
BOREHOLE_HEADER_URL = "https://www.bml3.nrw.de/service/bmlh"
BOREHOLE_DETAIL_URL = "https://www.bml3.nrw.de/service/bml"

# Coordinate reference systems
WKT_CRS = "EPSG:4326"       # WKT input is usually longitude/latitude
TARGET_CRS = "EPSG:25832"  # ETRS89 / UTM zone 32N, used by NRW WFS

# Depth discretization step in metres
STEP = 0.1


## 2. Define the area of interest

Use either **Option A: WKT polygon** or **Option B: shapefile path**.  
Set `AOI_MODE` to either `"wkt"` or `"shapefile"`.


In [2]:
AOI_MODE = "wkt"  # choose: "wkt" or "shapefile"

# Option A: WKT polygon in EPSG:4326
WKT_POLYGON = """
POLYGON ((
7.210851 51.486534,
7.217041 51.486534,
7.217041 51.488618,
7.210851 51.488618,
7.210851 51.486534
))
"""

# Option B: local shapefile path
SHAPEFILE_PATH = r"Insert your file path"


## 3. Helper functions


In [3]:
def bbox_from_wkt(wkt_polygon: str, wkt_crs: str = WKT_CRS, target_crs: str = TARGET_CRS):
    """Convert a WKT polygon to the target CRS and return its bounding box."""
    geom = shapely_wkt.loads(wkt_polygon)
    area_gdf = gpd.GeoDataFrame(geometry=[geom], crs=wkt_crs)
    area_gdf = area_gdf.to_crs(target_crs)
    return area_gdf.total_bounds, area_gdf


def bbox_from_shapefile(shapefile_path: str, target_crs: str = TARGET_CRS):
    """Read a shapefile, reproject it to the target CRS, and return its bounding box."""
    area_gdf = gpd.read_file(shapefile_path)

    if area_gdf.crs is None:
        raise ValueError("The shapefile has no CRS. Please define its CRS before using it.")

    area_gdf = area_gdf.to_crs(target_crs)
    return area_gdf.total_bounds, area_gdf


def force_point_geometry(geom):
    """Convert borehole geometry to point geometry when needed."""
    if geom is None or geom.is_empty:
        return None

    if geom.geom_type == "Point":
        return geom

    if geom.geom_type == "LineString":
        return Point(geom.coords[0])

    if geom.geom_type == "MultiLineString":
        first_line = list(geom.geoms)[0]
        return Point(first_line.coords[0])

    return geom.centroid


def get_text(parent, path, ns):
    """Safely extract text from an XML element."""
    elem = parent.find(path, ns)
    if elem is None or elem.text is None:
        return None
    return elem.text.strip()


## 4. Request borehole header data from the NRW WFS


In [4]:
if AOI_MODE == "wkt":
    bounds, area_gdf = bbox_from_wkt(WKT_POLYGON)
elif AOI_MODE == "shapefile":
    bounds, area_gdf = bbox_from_shapefile(SHAPEFILE_PATH)
else:
    raise ValueError("AOI_MODE must be either 'wkt' or 'shapefile'.")

xmin, ymin, xmax, ymax = bounds
print("Bounding box:", xmin, ymin, xmax, ymax)

params = {
    "SERVICE": "WFS",
    "VERSION": "2.0.0",
    "REQUEST": "GetFeature",
    "TYPENAMES": "bmlh:BoreholeHeader",
    "SRSNAME": TARGET_CRS,
    "BBOX": f"{xmin},{ymin},{xmax},{ymax},{TARGET_CRS}",
}

response = requests.get(BOREHOLE_HEADER_URL, params=params, timeout=60)
response.raise_for_status()

OUTPUT_GML.write_bytes(response.content)
print(f"Saved borehole headers to: {OUTPUT_GML}")

gdf_headers = gpd.read_file(OUTPUT_GML)
print(f"Downloaded boreholes: {len(gdf_headers)}")
gdf_headers.head()


Bounding box: 375778.18470545375 5705438.895674396 376213.5740864595 5705681.12391947
Saved borehole headers to: borehole_headers.gml
Downloaded boreholes: 14


c:\Users\INOJOSA\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: Field with same name (identifier) already exists in (BoreholeHeader). Skipping newer ones
  return ogr_read(


,gml_id,identifier,id,LanguageCode,LocalisedCharacterString,fullName|LocalisedCharacterString,databaseSource,totalLength,totalLength_uom,exportDate,...,scansTechAvail,samplesTechAvail,labDataLegalAvail,labDataTechAvail,originalLocation,locationMethod,elevation,levelMethod,levelMethod_,geometry
0,DABO_120940,DABO_120940,DABO_120940,deu,"259, Aufsl.327",DABO_120940,DABO,22.15,m,2024-11-26,...,T,N,AN,T,2584280 5706490,K,83.6,K,None,"LINESTRING (375910.41 5705463.67, 375910.41 57..."
1,DABO_121043,DABO_121043,DABO_121043,deu,None,DABO_121043,DABO,10.00,m,2024-11-25,...,T,N,AN,T,2584290 5706615,A,82.0,F,None,"LINESTRING (375925.52 5705588.11, 375925.52 57..."
2,DABO_121040,DABO_121040,DABO_121040,deu,3,DABO_121040,DABO,6.00,m,2024-11-25,...,T,N,AN,T,2584345 5706618,K,82.0,K,None,"LINESTRING (375980.58 5705588.86, 375980.58 57..."
3,DABO_120927,DABO_120927,DABO_120927,deu,"2, Aufsl.152",DABO_120927,DABO,12.00,m,2024-11-26,...,T,N,AN,T,2584165 5706520,K,84.0,K,None,"LINESTRING (375796.77 5705498.34, 375796.77 57..."
4,DABO_120941,DABO_120941,DABO_120941,deu,258 ;Aufsl.328,DABO_120941,DABO,23.20,m,2024-11-26,...,T,N,AN,T,2584310 5706545,K,83.4,K,None,"LINESTRING (375942.63 5705517.37, 375942.63 57..."


## 5. Convert borehole geometries to points


In [5]:
gdf_points = gdf_headers.copy()
gdf_points["geometry"] = gdf_points.geometry.apply(force_point_geometry)

gdf_points = gpd.GeoDataFrame(
    gdf_points,
    geometry="geometry",
    crs=gdf_headers.crs,
)

print(gdf_points.geometry.geom_type.value_counts())
gdf_points[["id", "geometry"]].head()


Point    14
Name: count, dtype: int64


,id,geometry
0,DABO_120940,POINT (375910.41 5705463.67)
1,DABO_121043,POINT (375925.52 5705588.11)
2,DABO_121040,POINT (375980.58 5705588.86)
3,DABO_120927,POINT (375796.77 5705498.34)
4,DABO_120941,POINT (375942.63 5705517.37)


## 6. Retrieve stratigraphic layers for each borehole


In [6]:
def parse_borehole_layers(feature_id: str) -> pd.DataFrame:
    """Download and parse BoreholeML stratigraphic layers for one borehole ID."""
    params = {
        "SERVICE": "WFS",
        "REQUEST": "GetFeature",
        "VERSION": "1.1.0",
        "TYPENAME": "Borehole",
        "featureID": feature_id,
        "outputFormat": "text/xml",
    }

    response = requests.get(BOREHOLE_DETAIL_URL, params=params, timeout=30)
    response.raise_for_status()

    root = ET.fromstring(response.content)

    ns = {
        "bml": "http://www.infogeo.de/boreholeml/3.0",
        "gmd": "http://www.isotc211.org/2005/gmd",
    }

    rows = []

    for layer in root.findall(".//bml:layer", ns):
        rows.append({
            "feature_id": feature_id,
            "from_m": get_text(layer, ".//bml:from", ns),
            "to_m": get_text(layer, ".//bml:to", ns),
            "rock_code": get_text(layer, ".//bml:rockCode", ns),
            "rock_name": get_text(layer, ".//bml:rockNameText/gmd:LocalisedCharacterString", ns),
        })

    df = pd.DataFrame(rows)

    for col in ["from_m", "to_m"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


In [7]:
borehole_ids = gdf_points["id"].dropna().tolist()

layer_tables = []
failed_ids = []

for feature_id in borehole_ids:
    try:
        layers_df = parse_borehole_layers(feature_id)

        if layers_df.empty:
            failed_ids.append(feature_id)
            continue

        layer_tables.append(layers_df)

    except Exception as exc:
        print(f"Skipped {feature_id}: {exc}")
        failed_ids.append(feature_id)

all_layers = pd.concat(layer_tables, ignore_index=True) if layer_tables else pd.DataFrame()

print(f"Successful boreholes: {len(layer_tables)}")
print(f"Skipped boreholes: {len(failed_ids)}")
print(f"Stratigraphic layer records: {len(all_layers)}")

all_layers.head()


Successful boreholes: 14
Skipped boreholes: 0
Stratigraphic layer records: 117


,feature_id,from_m,to_m,rock_code,rock_name
0,DABO_120940,0.0,2.40,"AY,AG","Bauschutt / Straßenaufbruch, Erdaushub"
1,DABO_120940,2.4,4.90,"U,FS,2,T,2","Schluff, schwach feinsandig, schwach tonig, br..."
2,DABO_120940,14.9,22.15,TM,"Tonmergel, fest, hart, schwach klüftig"
3,DABO_120940,4.9,7.50,"U,FS,2,T,2","Schluff, schwach feinsandig, schwach tonig, he..."
4,DABO_120940,7.5,11.50,"U,FS,3","Schluff, feinsandig, hellgrau, graubraun"


## 7. Expand stratigraphy to regular depth intervals

The original BoreholeML layers may have variable thickness. This step converts them into regular intervals, for example 10 cm layers. This makes the data easier to visualize in 3D.


In [8]:
def expand_layers_to_fixed_grid(df: pd.DataFrame, step: float = 0.1) -> pd.DataFrame:
    """Expand variable-thickness stratigraphic layers into regular depth intervals."""
    rows = []

    for _, row in df.dropna(subset=["from_m", "to_m"]).iterrows():
        start_i = int(np.floor(row["from_m"] / step))
        end_i = int(np.ceil(row["to_m"] / step))

        for i in range(start_i, end_i):
            depth_from = round(i * step, 2)
            depth_to = round((i + 1) * step, 2)
            depth_mid = round((depth_from + depth_to) / 2, 2)

            # Keep only intervals overlapping the original layer
            if depth_to <= row["from_m"] or depth_from >= row["to_m"]:
                continue

            rows.append({
                "id": row["feature_id"],
                "depth_from_m": depth_from,
                "depth_to_m": depth_to,
                "depth_mid_m": depth_mid,
                "rock_code": row["rock_code"],
                "rock_name": row["rock_name"],
            })

    return pd.DataFrame(rows)


layers_fixed = expand_layers_to_fixed_grid(all_layers, step=STEP)

layers_3d_gdf = layers_fixed.merge(
    gdf_points[["id", "geometry"]],
    on="id",
    how="left",
)

layers_3d_gdf = gpd.GeoDataFrame(
    layers_3d_gdf,
    geometry="geometry",
    crs=gdf_points.crs,
)

print(f"Expanded layer records: {len(layers_3d_gdf)}")
layers_3d_gdf.head()


Expanded layer records: 3420


,id,depth_from_m,depth_to_m,depth_mid_m,rock_code,rock_name,geometry
0,DABO_120940,0.0,0.1,0.05,"AY,AG","Bauschutt / Straßenaufbruch, Erdaushub",POINT (375910.41 5705463.67)
1,DABO_120940,0.1,0.2,0.15,"AY,AG","Bauschutt / Straßenaufbruch, Erdaushub",POINT (375910.41 5705463.67)
2,DABO_120940,0.2,0.3,0.25,"AY,AG","Bauschutt / Straßenaufbruch, Erdaushub",POINT (375910.41 5705463.67)
3,DABO_120940,0.3,0.4,0.35,"AY,AG","Bauschutt / Straßenaufbruch, Erdaushub",POINT (375910.41 5705463.67)
4,DABO_120940,0.4,0.5,0.45,"AY,AG","Bauschutt / Straßenaufbruch, Erdaushub",POINT (375910.41 5705463.67)


## 8. Export processed layers


In [9]:
layers_3d_gdf.to_file(OUTPUT_GPKG, driver="GPKG")
print(f"Saved processed stratigraphy to: {OUTPUT_GPKG}")


Saved processed stratigraphy to: borehole_layers_3d.gpkg


## 9. Create an interactive 3D Plotly visualization


In [10]:
gdf3d = layers_3d_gdf.copy()

# Extract coordinates and define depth as negative z values
gdf3d["x"] = gdf3d.geometry.x
gdf3d["y"] = gdf3d.geometry.y
gdf3d["z"] = -gdf3d["depth_mid_m"]

# Optional: sample for faster plotting if the dataset is very large
# gdf3d = gdf3d.sample(min(50000, len(gdf3d)), random_state=42)

fig = px.scatter_3d(
    gdf3d,
    x="x",
    y="y",
    z="z",
    color="rock_name",
    hover_data=[
        "id",
        "depth_from_m",
        "depth_to_m",
        "depth_mid_m",
        "rock_code",
        "rock_name",
    ],
    opacity=0.7,
    height=800,
)

fig.update_traces(marker=dict(size=2))

fig.update_layout(
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Depth below ground surface (m)",
    )
)

fig.show()


## 10. Save the 3D visualization as HTML


In [ ]:
fig.write_html(OUTPUT_HTML, auto_open=False)
print(f"Saved 3D visualization to: {OUTPUT_HTML}")


## Notes

- The WFS request uses the bounding box of the WKT polygon or shapefile. For rectangular areas this is usually sufficient.
- For irregular polygons, an additional spatial clipping step can be added after downloading the boreholes.
- Large areas may return many boreholes and generate a very large 3D dataset after depth discretization.
- If the Plotly figure does not appear inside the notebook, export it to HTML and open it in a browser.
